# Final Submission Notebook

Running this notebook is enough to reproduce the final project outputs from the raw CSV files. It builds features in memory, validates and trains the CatBoost model, writes `outputs/submission.csv`, saves evaluation tables, and generates the report figures. It does not write intermediate processed datasets or a saved model artifact.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "data" / "train.csv").exists():
    project_root = project_root.parent

config_dir = project_root / "config"
for path in [project_root, config_dir]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

data_dir = project_root / "data"
outputs_dir = project_root / "outputs"

print("project root:", project_root)

In [ ]:
# Install missing dependencies into the active notebook kernel if needed.
import subprocess

required_packages = ["catboost", "sklearn", "matplotlib", "seaborn"]

def missing_required_packages():
    missing = []
    for package_name in required_packages:
        try:
            __import__(package_name)
        except ModuleNotFoundError:
            missing.append(package_name)
    return missing

missing_packages = missing_required_packages()

if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(project_root / "config" / "requirements.txt"),
    ])

missing_packages = missing_required_packages()
if missing_packages:
    raise ModuleNotFoundError(
        "Still missing packages after installation: " + ", ".join(missing_packages)
    )

print("All required packages are available.")


## Build Processed Data

The raw feature columns come from `config/model_columns.txt`. `Closed Date` is not included as an input feature; it is used only to create the target.

In [ ]:
from preprocessing import build_processed_datasets, load_column_list

input_columns = load_column_list(project_root / "config" / "model_columns.txt")

X_train, y_train, X_test, metadata = build_processed_datasets(
    data_dir=data_dir,
    input_columns=input_columns,
)

print("training rows:", len(X_train))
print("test rows:", len(X_test))
print("input columns:", len(metadata["input_columns"]))
print("final features:", len(metadata["output_features"]))
print("categorical features:", len(metadata["categorical_cols"]))


## Validate Final Model

The official evaluation metric is accuracy. A stratified holdout split is used for the main validation score, and five-fold stratified cross-validation is run to check score stability.

In [ ]:
from train_and_evaluate_model import (
    DEFAULT_PARAMS_FILE,
    build_model,
    cross_validate_model,
    evaluate_train_valid_split,
    load_model_params,
    save_outputs,
)

model_params = load_model_params(DEFAULT_PARAMS_FILE)
categorical_cols = metadata["categorical_cols"]

split_result = evaluate_train_valid_split(
    X_train,
    y_train,
    categorical_cols,
    model_params,
)

cv_results, cv_summary = cross_validate_model(
    X_train,
    y_train,
    categorical_cols,
    model_params,
)
outputs_dir.mkdir(exist_ok=True)
cv_results.to_csv(outputs_dir / "cross_validation_results.csv", index=False)

print("training accuracy:", round(split_result["training_accuracy"], 4))
print("validation accuracy:", round(split_result["validation_accuracy"], 4))
print("train-validation gap:", round(split_result["training_accuracy"] - split_result["validation_accuracy"], 4))
print("cv mean validation accuracy:", round(cv_summary["mean_validation_accuracy"], 4))
print("cv validation std:", round(cv_summary["std_validation_accuracy_sample"], 4))

## Train On All Labeled Rows And Save Submission

After validation, the final CatBoost model is fitted on all labeled training rows and used to predict the provided test file.

In [ ]:
final_model = build_model(model_params)
final_model.fit(X_train, y_train, cat_features=categorical_cols)
test_predictions = final_model.predict(X_test).astype(int)

submission_template = pd.read_csv(data_dir / "submission.csv")

summary, report, confusion = save_outputs(
    outputs_dir=outputs_dir,
    submission=submission_template,
    test_predictions=test_predictions,
    validation_predictions=split_result["validation_predictions"],
    y_valid=split_result["y_valid"],
    training_accuracy=split_result["training_accuracy"],
    validation_accuracy=split_result["validation_accuracy"],
    cv_summary=cv_summary,
    metadata=metadata,
)

display(summary)
display(report)
display(confusion)

final_submission = pd.read_csv(outputs_dir / "submission.csv")
prediction_col = "prediction" if "prediction" in final_submission.columns else final_submission.columns[-1]

assert list(final_submission.columns) == list(submission_template.columns)
assert len(final_submission) == len(pd.read_csv(data_dir / "test.csv"))
assert final_submission[prediction_col].isin([0, 1]).all()
assert final_submission[prediction_col].notna().all()

print("saved:", outputs_dir / "submission.csv")
print("prediction counts:")
print(final_submission[prediction_col].value_counts().sort_index())
display(final_submission.head())

## Generate Report Figures

The final report references four figures. This cell creates exactly those images from the saved data and validation outputs.


In [ ]:
from create_report_figures import main as create_report_figures

create_report_figures()

expected_figures = {
    "01_target_distribution.png",
    "06_requests_by_creation_hour.png",
    "09_feature_set_summary.png",
    "13_confusion_matrix_heatmap.png",
}
created_figures = {path.name for path in (outputs_dir / "figures").glob("*.png")}
assert created_figures == expected_figures

print("report figures created:")
for figure_name in sorted(created_figures):
    print("-", outputs_dir / "figures" / figure_name)


## Final Output Checklist

This final check lists the files produced by the notebook.


In [ ]:
expected_outputs = [
    outputs_dir / "submission.csv",
    outputs_dir / "model_summary.csv",
    outputs_dir / "classification_report.csv",
    outputs_dir / "confusion_matrix.csv",
    outputs_dir / "cross_validation_results.csv",
    outputs_dir / "figures" / "01_target_distribution.png",
    outputs_dir / "figures" / "06_requests_by_creation_hour.png",
    outputs_dir / "figures" / "09_feature_set_summary.png",
    outputs_dir / "figures" / "13_confusion_matrix_heatmap.png",
]

missing_outputs = [path for path in expected_outputs if not path.exists()]
assert not missing_outputs, missing_outputs

for path in expected_outputs:
    print(path)
